In [3]:
from pyspark.sql import functions as F

StatementMeta(, 937a8906-b8be-4323-a66e-5973c9dfc3bb, 5, Finished, Available, Finished, False)

In [4]:
customers_df = spark.read.table("silver_customers")
employees_df = spark.read.table("silver_employees")
categories_df = spark.read.table("silver_categories")
departments_df = spark.read.table("silver_departments")
products_df = spark.read.table("silver_products")
shippers_df = spark.read.table("silver_shippers")
orders_df = spark.read.table("silver_orders")
order_details_df = spark.read.table("silver_order_details")
payments_df = spark.read.table("silver_payments")
shipments_df = spark.read.table("silver_shipments")


# let's check
# print("All Schemas")

# customers_df.printSchema()
# employees_df.printSchema()
# departments_df.printSchema()
# products_df.printSchema()
# categories_df.printSchema()
# shippers_df.printSchema()
# orders_df.printSchema()
# order_details_df.printSchema()
# payments_df.printSchema()
# shipments_df.printSchema()

StatementMeta(, 937a8906-b8be-4323-a66e-5973c9dfc3bb, 6, Finished, Available, Finished, False)

In [5]:
# --- Dim_date ---
dim_date = spark.sql("""
    SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) as FullDate
""").select(
    F.date_format("FullDate", "yyyyMMdd").cast("int").alias("DateKey"),
    "FullDate",
    F.dayofmonth("FullDate").alias("Day"),
    F.month("FullDate").alias("Month"),
    F.date_format("FullDate", "MMMM").alias("MonthName"),
    F.quarter("FullDate").alias("Quarter"),
    F.year("FullDate").alias("Year"),
    F.date_format("FullDate", "EEEE").alias("DayOfWeek"),
    F.when(F.dayofweek("FullDate").isin(6, 7), True).otherwise(False).alias("IsWeekend")
)

# --- Dim_Customers ---
dim_customers = customers_df

# --- Dim_Employees (Join with Departments) ---
dim_employees = employees_df.alias("e") \
    .join(departments_df.alias("d"), "DepartmentID", "left") \
    .select("e.EmployeeID", "e.ManagerID", "e.DepartmentID",
            "d.DepartmentName", "e.FirstName", "e.LastName", "e.Gender", "e.Salary", "e.HireDate")

# --- Dim_Products (Join with Categories) ---
dim_products = products_df.alias("p") \
    .join(categories_df.alias("c"), "CategoryID", "left") \
    .select("p.ProductID", "p.ProductName", "p.Brand", "p.CategoryID",
            "c.CategoryName", "c.MainCategory", "p.Price", "p.Cost", "p.StockType",
            "p.warranty_months", "p.color")

# --- Dim_Shippers ---
dim_shippers = shippers_df

# let's check
# print("All dims Schemas")

# dim_shippers.printSchema()
# dim_products.printSchema()
# dim_customers.printSchema()
# dim_date.printSchema()
# dim_employees.printSchema()

# print("Done")

StatementMeta(, 937a8906-b8be-4323-a66e-5973c9dfc3bb, 7, Finished, Available, Finished, False)

In [6]:
# --- Fact_Order_Details ---
fact_order_details = order_details_df.alias("od") \
    .join(orders_df.alias("o"), "OrderID", "inner") \
    .withColumn("OrderDateKey", F.date_format("o.OrderDate", "yyyyMMdd").cast("int")) \
    .select("od.OrderDetailID", "OrderID", "OrderDateKey",
            F.col("o.CustomerID"), F.col("o.SalesEmployeeID").alias("EmployeeID"),
            "od.ProductID", F.col("o.Status").alias("OrderStatus"),
            "od.Quantity", "od.UnitPrice", "od.Discount", "od.TotalSales")

# --- Fact_Payments ---
fact_payments = payments_df.alias("p") \
    .join(orders_df.alias("o"), "OrderID", "inner") \
    .withColumn("PaymentDateKey", F.date_format("p.PaymentDate", "yyyyMMdd").cast("int")) \
    .select("p.PaymentID", "OrderID", "PaymentDateKey", F.col("o.CustomerID"),
            "p.PaymentMethod", "p.Amount", "p.PaymentStatus")

# --- Fact_Shipments ---
fact_shipments = shipments_df.alias("s") \
    .join(orders_df.alias("o"), "OrderID", "inner") \
    .withColumn("ShipDateKey", F.date_format("s.ShipDate", "yyyyMMdd").cast("int")) \
    .withColumn("DeliveryDateKey", F.date_format("s.DeliveryDate", "yyyyMMdd").cast("int")) \
    .select("s.ShipmentID", "OrderID", "ShipDateKey", "DeliveryDateKey",
            F.col("o.CustomerID"), "s.ShipperID", "s.DaysToArrive", "s.DeliveryStatus")

# let's check
# print("All facts Schemas")

# fact_shipments.printSchema()
# fact_order_details.printSchema()
# fact_payments.printSchema()

# print("Done")

StatementMeta(, 937a8906-b8be-4323-a66e-5973c9dfc3bb, 8, Finished, Available, Finished, False)

In [7]:
def write_to_gold(df, table_name):
    
    (df.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(table_name))
     
    # print(f"Table {table_name} written to Gold successfully.")

write_to_gold(dim_date, "dim_date")
write_to_gold(dim_customers, "dim_customers")
write_to_gold(dim_employees, "dim_employees")
write_to_gold(dim_products, "dim_products")
write_to_gold(dim_shippers, "dim_shippers")

write_to_gold(fact_order_details, "fact_order_details")
write_to_gold(fact_payments, "fact_payments")
write_to_gold(fact_shipments, "fact_shipments")

# print("Good bye fabric allah y7ra2k")

StatementMeta(, 937a8906-b8be-4323-a66e-5973c9dfc3bb, 9, Finished, Available, Finished, False)

Table dim_date written to Gold successfully.
Table dim_customers written to Gold successfully.
Table dim_employees written to Gold successfully.
Table dim_products written to Gold successfully.
Table dim_shippers written to Gold successfully.
Table fact_order_details written to Gold successfully.
Table fact_payments written to Gold successfully.
Table fact_shipments written to Gold successfully.
